# 04 — Mining aliases from the training ground truth
Finds token substitutions that recur between matched records (e.g. `bangalore ↔ bengaluru`,
`corp ↔ corporation`). Uses **only the provided training data** — no external lookups.
Promising aliases can be added to the dictionaries in `src/normalize.py`.

In [ ]:
# --- setup: run from the repo root or from notebooks/ ---
import sys, os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
# pick the first data dir that exists; full data is ~12M rows/split, so EDA defaults to samples
for cand in ["dataset_sample", "dataset_region", "dataset", "fake_dataset"]:
    if (ROOT / cand / "train").exists():
        DATA = ROOT / cand
        break
CACHE = ROOT / "cache"                  # pipeline cache (blocking .npz, oof_train.pkl)
CACHE = ROOT / "cache"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("data:", DATA)

In [ ]:
from data import load_split, load_ground_truth
from sims import jaro_winkler
from collections import Counter
truth = load_ground_truth(DATA)
s1, s23 = load_split(DATA, "train")
A = s1.set_index("entity_id"); B = s23.set_index("entity_id")
pairs = [(a, b) for a, ms in truth.items() for b in ms if a in A.index and b in B.index]
print(len(pairs), "matched pairs")

In [ ]:
def token_swaps(col):
    """Count (token_a, token_b) where the two records differ by exactly one token each."""
    c = Counter()
    for a, b in pairs:
        ta, tb = set(A.at[a, col].split()), set(B.at[b, col].split())
        da, db = ta - tb, tb - ta
        if len(da) == 1 and len(db) == 1:
            c[tuple(sorted((next(iter(da)), next(iter(db)))))] += 1
    return c

def as_table(c, min_count=3):
    rows = [dict(a=x, b=y, count=n, jw=round(jaro_winkler(x, y), 3)) for (x, y), n in c.most_common() if n >= min_count]
    return pd.DataFrame(rows)

name_sw = as_table(token_swaps("name_core"))
addr_sw = as_table(token_swaps("addr_n"))
print("name swaps that are NOT simple typos (jw < 0.85):")
display(name_sw[name_sw.jw < 0.85].head(40))

In [ ]:
print("address swaps that are NOT simple typos (jw < 0.85) -> transliterations / abbreviations:")
display(addr_sw[addr_sw.jw < 0.85].head(40))

In [ ]:
# save for review; move the good ones into src/normalize.py by hand
CACHE.mkdir(exist_ok=True)
out = {"name": name_sw[name_sw.jw < 0.85].to_dict("records"), "addr": addr_sw[addr_sw.jw < 0.85].to_dict("records")}
json.dump(out, open(CACHE / "alias_candidates.json", "w"), indent=1)
print("saved", CACHE / "alias_candidates.json")